In [19]:
!pip -q install pypdf faiss-cpu langchain_community langchain_text_splitters langchain_openai langchain_huggingface

In [20]:
import os
import requests
from pathlib import Path
from getpass import getpass

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

In [8]:
DATA_DIR = Path("data")
INDEX_DIR = Path("faiss_index")

DATA_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)

PDF_URLS = [
    "https://arxiv.org/pdf/1706.03762.pdf",
    "https://arxiv.org/pdf/1810.04805.pdf",
    "https://arxiv.org/pdf/2005.11401.pdf"
]

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "google/gemini-2.5-flash"

TOP_K = 5

### Download PDF files

In [9]:
def download_file(url, save_path):
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    
    with open(save_path, "wb") as f:
        f.write(response.content)


for url in PDF_URLS:
    file_name = url.split("/")[-1]
    save_path = DATA_DIR / file_name
    
    if not save_path.exists():
        download_file(url, save_path)

sorted([p.name for p in DATA_DIR.glob("*.pdf")])

['1706.03762.pdf', '1810.04805.pdf', '2005.11401.pdf']

### Load PDF documents

In [10]:
all_docs = []

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    
    for doc in docs:
        doc.metadata["file_name"] = pdf_path.name
        doc.metadata["page_number"] = int(doc.metadata.get("page", 0)) + 1
    all_docs.extend(docs)
    
len(all_docs), all_docs[0].metadata, all_docs[0].page_content[:100]

(50,
 {'producer': 'pdfTeX-1.40.25',
  'creator': 'LaTeX with hyperref',
  'creationdate': '2024-04-10T21:11:43+00:00',
  'author': '',
  'keywords': '',
  'moddate': '2024-04-10T21:11:43+00:00',
  'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
  'subject': '',
  'title': '',
  'trapped': '/False',
  'source': 'data/1706.03762.pdf',
  'total_pages': 15,
  'page': 0,
  'page_label': '1',
  'file_name': '1706.03762.pdf',
  'page_number': 1},
 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and')

### Split documents into chunks

In [11]:
spliter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

chunks = spliter.split_documents(all_docs)

In [12]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i+1
    
len(chunks), chunks[0].metadata, chunks[0].page_content[:200]

(184,
 {'producer': 'pdfTeX-1.40.25',
  'creator': 'LaTeX with hyperref',
  'creationdate': '2024-04-10T21:11:43+00:00',
  'author': '',
  'keywords': '',
  'moddate': '2024-04-10T21:11:43+00:00',
  'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
  'subject': '',
  'title': '',
  'trapped': '/False',
  'source': 'data/1706.03762.pdf',
  'total_pages': 15,
  'page': 0,
  'page_label': '1',
  'file_name': '1706.03762.pdf',
  'page_number': 1,
  'chunk_id': 1},
 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\n')

### Build or load FAISS vector index

In [21]:
embeddings = HuggingFaceEmbeddings()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
vector_store = FAISS.from_documents(chunks, embeddings)

vector_store.save_local(str(INDEX_DIR))
vector_store.index.ntotal

184

### Create the document QA agent functions

In [ ]:
llm = ChatOpenAI(model=CHAT_MODEL)

In [24]:
def rewrite_question(question):
    prompt = f"""
    You turn a user question into a short retrieval-friendly search query.
    Keep important keywords, entities, topics, and technical phrases.
    Return only one line.

    User question: {question}
    """
    return llm.invoke(prompt).content.strip()

In [25]:
def retrieve_chunks(question, k=TOP_K):
    search_query = rewrite_question(question)
    docs = vector_store.similarity_search(search_query, k=k)
    return search_query, docs


def make_context(docs):
    context_parts = []
    source_lines = []
    
    for i, doc in enumerate(docs, start=1):
        file_name = doc.metadata.get("file_name", "unknown")
        page_number = doc.metadata.get("page_number", "unknown")
        chunk_id = doc.metadata.get("chunk_id", "unknown")
        text = " ".join(doc.page_content.split())
        
        context_parts.append(
            f"[{i}] File: {file_name} | Page: {page_number} | Chunk: {chunk_id}\n{text}"
        )
        source_lines.append(
            f"[{i}] {file_name} | page {page_number} | chunk {chunk_id}"
        )
    
    context = "\n\n".join(context_parts)
    return context, source_lines

In [26]:
def answer_question(question, k=TOP_K):
    search_query, docs = retrieve_chunks(question, k=k)
    context, source_lines = make_context(docs)
    
    prompt = f"""
    You are a careful document question answering agent.

    Rules:
    1. Use only the context below.
    2. If the answer is missing, say you do not have enough evidence from the documents.
    3. After important claims, add citations like [1] or [2].
    4. Do not invent facts.
    5. Keep the answer clear and direct.

    Question:
    {question}

    Context:
    {context}
    """
    
    answer = llm.invoke(prompt).content.strip()
    
    return {
        "question": question,
        "search_query": search_query,
        "answer": answer,
        "sources": source_lines,
        "docs": docs
    }

### Test the agent with a sample question

In [27]:
question = "Why does the Transformer paper say self-attention is useful?"
result = answer_question(question, k=5)

print("Question:")
print(result["question"], "\n")
print("Search Query:")
print(result["search_query"], "\n")
print("Answer:")
print(result["answer"], "\n")

print("Sources:")
for item in result["sources"]:
    print(item)

Question:
Why does the Transformer paper say self-attention is useful? 

Search Query:
Transformer paper self-attention usefulness advantages 

Answer:
The Transformer paper indicates that self-attention is useful for several reasons:

1.  It reduces the number of operations required to relate signals from two arbitrary input or output positions to a constant number of operations. This addresses the difficulty other models, like ConvS2S and ByteNet, have in learning dependencies between distant positions, where operations grow linearly or logarithmically with distance [3].
2.  Self-attention, also known as intra-attention, relates different positions of a single sequence to compute a representation of that sequence [3].
3.  It has been successfully applied in various tasks, including reading comprehension and abstractive summarization [3].
4.  Multi-head attention, a component of the self-attention mechanism, allows the model to jointly attend to information from different representati

In [ ]:
question = "How is BERT pre-training different from left-to-right language modeling?"
result = answer_question(question, k=5)

print(result["answer"], "\n")
for item in result["sources"]:
    print(item)

BERT's pre-training differs from left-to-right language modeling in several key ways:

1.  **Bidirectionality** BERT does not use traditional left-to-right or right-to-left language models for pre-training. Instead, it is a deep bidirectional model where representations are jointly conditioned on both left and right context across all layers [1, 2]. In contrast, standard left-to-right language models (like OpenAI GPT) are unidirectional, conditioning only on the left context [1, 2, 3].
2.  **Pre-training Tasks** BERT is pre-trained using two unsupervised tasks: Masked Language Model (MLM) and Next Sentence Prediction (NSP) [1]. A "LTR & No NSP" model, which is like OpenAI GPT, is trained as a standard Left-to-Right (LTR) LM and only conditions on the left context [3]. The Masked LM objective specifically allows for the deep bidirectionality [1].
3.  **Prediction Strategy** Left-to-right models predict every token sequentially [5]. BERT's Masked LM, on the other hand, predicts only a pe

### Run an interactive question loop

In [ ]:
while True:
    question = input("Ask a question: ").strip()

    if question.lower() in ["exit", "quit", "stop"]:
        break

    result = answer_question(question, k=5)

    print()
    print(result["answer"], "\n")
    for item in result["sources"]:
        print(item)
    print()